In [ ]:
#Imports..

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

from torchvision.models import (
    resnet34,
    ResNet34_Weights
)

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
# Device..
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
# Dataset..

mean = (0.4914, 0.4822, 0.4465)
std = (0.247, 0.243, 0.261)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_dataset = torchvision.datasets.CIFAR10(
    root=r'./data',
    train=True,
    download=False,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=r'./data',
    train=False,
    download=False,
    transform=test_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("Dataset Loaded")
print(len(train_dataset))
print(len(test_dataset))

In [ ]:
# Gauseian Blur..

class GaussianBlur(nn.Module):

    def __init__(self, channels):

        super(GaussianBlur, self).__init__()

        kernel = torch.tensor([
            [1., 2., 1.],
            [2., 4., 2.],
            [1., 2., 1.]
        ])

        kernel /= 16.0

        kernel = kernel.view(1,1,3,3)

        kernel = kernel.repeat(channels,1,1,1)

        self.weight = nn.Parameter(
            kernel,
            requires_grad=False
        )

        self.groups = channels

    def forward(self, x):

        return F.conv2d(
            x,
            self.weight,
            padding=1,
            groups=self.groups
        )

In [ ]:
# Denoise Block..

class DenoiseBlock(nn.Module):

    def __init__(self, channels):

        super(DenoiseBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):

        residual = x

        out = F.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        out = self.bn2(
            self.conv2(out)
        )

        out += residual

        out = F.relu(out)

        return out

In [ ]:
# AvgMAxPool

class AvgMaxPool(nn.Module):

    def __init__(self):

        super(AvgMaxPool, self).__init__()

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.maxpool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):

        avg = self.avgpool(x)

        max_ = self.maxpool(x)

        return avg + max_

In [ ]:
# Pretrained Securenet.

class SecureResNet34(nn.Module):

    def __init__(self, num_classes=10):

        super(SecureResNet34, self).__init__()

        self.backbone = resnet34(
            weights=ResNet34_Weights.DEFAULT
        )

        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()

        self.gaussian = GaussianBlur(64)

        self.denoise = DenoiseBlock(512)

        self.pool = AvgMaxPool()

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):

        x = self.backbone.conv1(x)

        x = self.backbone.bn1(x)

        x = self.backbone.relu(x)

        x = self.gaussian(x)

        x = self.backbone.layer1(x)

        x = self.backbone.layer2(x)

        x = self.backbone.layer3(x)

        x = self.backbone.layer4(x)

        x = self.denoise(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [ ]:
# Creating Model..

model = SecureResNet34().to(device)

print("Pretrained SecureResNet34 Ready")

In [ ]:
# Loss + optimizer..

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15
)

print("Optimizer Ready")

In [ ]:
#Fast PGD3 Attack..

def pgd_attack(model,
               images,
               labels,
               eps=8/255,
               alpha=2/255,
               steps=3):

    images = images.clone().detach().to(device)

    labels = labels.to(device)

    loss = nn.CrossEntropyLoss()

    adv_images = images.clone().detach()

    adv_images = adv_images + torch.empty_like(
        adv_images
    ).uniform_(-eps, eps)

    adv_images = torch.clamp(
        adv_images,
        min=-1,
        max=1
    )

    for _ in range(steps):

        adv_images.requires_grad = True

        outputs = model(adv_images)

        cost = loss(outputs, labels)

        grad = torch.autograd.grad(
            cost,
            adv_images,
            retain_graph=False,
            create_graph=False
        )[0]

        adv_images = adv_images.detach() + alpha * grad.sign()

        delta = torch.clamp(
            adv_images - images,
            min=-eps,
            max=eps
        )

        adv_images = torch.clamp(
            images + delta,
            min=-1,
            max=1
        ).detach()

    return adv_images

In [ ]:
#training..

epochs = 35

best_acc = 0

for epoch in range(epochs):

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    loop = tqdm(train_loader)

    for images, labels in loop:

        images = images.to(device)

        labels = labels.to(device)

        adv_images = pgd_attack(
            model,
            images,
            labels,
            steps=3
        )

        mixed_images = torch.cat(
            [images, adv_images],
            dim=0
        )

        mixed_labels = torch.cat(
            [labels, labels],
            dim=0
        )

        optimizer.zero_grad()

        outputs = model(mixed_images)

        loss = criterion(
            outputs,
            mixed_labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += mixed_labels.size(0)

        correct += predicted.eq(
            mixed_labels
        ).sum().item()

        loop.set_description(
            f"Epoch [{epoch+1}/{epochs}]"
        )

    scheduler.step()

    acc = 100. * correct / total

    avg_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}")

    print(f"Loss: {avg_loss:.4f}")

    print(f"Accuracy: {acc:.2f}%")

    if acc > best_acc:

        best_acc = acc

        torch.save(
            model.state_dict(),
            "best_secure_resnet34_pretrained_pgd3.pth"
        )

        print("Best model saved.")

    if (epoch + 1) % 5 == 0:

        torch.save(
            model.state_dict(),
            f"secure_resnet34_pretrained_pgd3_epoch_{epoch+1}.pth"
  
        )